In [38]:
import pandas as pd
df = pd.read_csv("../dataset/raw/occupancy.csv")
df

,time,102,104,105,106,107,108,109,110,111,...,1156,1159,1162,1163,1164,1166,1167,1168,1172,1173
0,2022-09-01 00:00:00,17.0,1.0,15.0,54.0,23.0,13.0,11.0,6.0,20.0,...,10.0,31.0,9.0,2.0,45.0,26.0,183.0,17.0,2.0,10.0
1,2022-09-01 01:00:00,16.0,1.0,19.0,50.0,25.0,13.0,7.0,12.0,19.0,...,7.0,31.0,8.0,1.0,42.0,25.0,174.0,12.0,2.0,9.0
2,2022-09-01 02:00:00,16.0,1.0,20.0,44.0,24.0,16.0,12.0,6.0,21.0,...,7.0,38.0,9.0,2.0,48.0,24.0,196.0,13.0,2.0,12.0
3,2022-09-01 03:00:00,16.0,1.0,20.0,44.0,22.0,16.0,12.0,6.0,21.0,...,7.0,38.0,9.0,2.0,48.0,24.0,187.0,13.0,2.0,13.0
4,2022-09-01 04:00:00,17.0,1.0,20.0,50.0,24.0,13.0,8.0,12.0,20.0,...,7.0,31.0,8.0,1.0,42.0,25.0,167.0,11.0,2.0,10.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4339,2023-02-28 19:00:00,10.0,2.0,15.0,43.0,13.0,12.0,6.0,9.0,12.0,...,4.0,37.0,2.0,5.0,33.0,22.0,145.0,8.0,3.0,9.0
4340,2023-02-28 20:00:00,13.0,2.0,16.0,43.0,16.0,14.0,8.0,9.0,13.0,...,4.0,38.0,3.0,5.0,30.0,23.0,171.0,10.0,4.0,10.0
4341,2023-02-28 21:00:00,14.0,2.0,16.0,44.0,18.0,13.0,7.0,10.0,14.0,...,3.0,36.0,7.0,5.0,34.0,27.0,183.0,10.0,4.0,10.0
4342,2023-02-28 22:00:00,14.0,2.0,18.0,45.0,18.0,13.0,7.0,11.0,15.0,...,3.0,35.0,8.0,4.0,35.0,29.0,197.0,10.0,4.0,10.0


In [39]:
occupancy_long = df.melt(
    id_vars=["time"],
    value_vars=df.columns[1:],
    var_name="zone",
    value_name="occupancy"
)
occupancy_long

,time,zone,occupancy
0,2022-09-01 00:00:00,102,17.0
1,2022-09-01 01:00:00,102,16.0
2,2022-09-01 02:00:00,102,16.0
3,2022-09-01 03:00:00,102,16.0
4,2022-09-01 04:00:00,102,17.0
...,...,...,...
1194595,2023-02-28 19:00:00,1173,9.0
1194596,2023-02-28 20:00:00,1173,10.0
1194597,2023-02-28 21:00:00,1173,10.0
1194598,2023-02-28 22:00:00,1173,10.0


In [40]:
occupancy_long.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1194600 entries, 0 to 1194599
Data columns (total 3 columns):
 #   Column     Non-Null Count    Dtype  
---  ------     --------------    -----  
 0   time       1194600 non-null  object 
 1   zone       1194600 non-null  object 
 2   occupancy  1194600 non-null  float64
dtypes: float64(1), object(2)
memory usage: 27.3+ MB


In [41]:
occupancy_long["time"] = pd.to_datetime(
    occupancy_long["time"]
)

In [42]:
occupancy_long.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1194600 entries, 0 to 1194599
Data columns (total 3 columns):
 #   Column     Non-Null Count    Dtype         
---  ------     --------------    -----         
 0   time       1194600 non-null  datetime64[ns]
 1   zone       1194600 non-null  object        
 2   occupancy  1194600 non-null  float64       
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 27.3+ MB


In [43]:
zone_avg = (
    occupancy_long
    .groupby("zone")["occupancy"]
    .mean()
    .sort_values(ascending=False)
)

zone_avg.head(10)

zone
1167    137.304558
526     102.887891
519      87.164825
1011     82.056860
703      80.162178
335      76.468692
1049     74.671961
633      70.684162
851      65.610497
1068     64.189572
Name: occupancy, dtype: float64

In [44]:
#根据UrbanEV数据中的occupancy指标计算，各区域存在明显需求压力差异。其中1167区域平均occupancy最高，达到137.30，说明该区域在研究周期内持续存在较高充电需求或设施利用压力，可能需要进一步评估充电设施扩容需求。

In [45]:
high_pressure = (
    occupancy_long[occupancy_long["occupancy"] > 100]
    .groupby("zone")
    .size()
    .sort_values(ascending=False)
)

high_pressure.head(10)

zone
1167    3317
526     2375
519     1772
1049    1214
703      801
1011     576
851      412
633      146
335      121
595       97
dtype: int64

In [46]:
#一天x点压力最高时刻
hour_avg = (
    occupancy_long
    .groupby(
        occupancy_long["time"].dt.hour
    )["occupancy"]
    .mean()
)

hour_avg.index.name = "hour"


In [47]:
zone_1167_hour = (
    occupancy_long[
        occupancy_long["zone"]=="1167"
    ]
    .groupby(
        occupancy_long["time"].dt.hour
    )["occupancy"]
    .mean()
)

zone_1167_hour

time
0     173.419890
1     179.944751
2     173.425414
3     171.817680
4     168.734807
5     172.917127
6     174.917127
7     169.767956
8     141.154696
9     111.580110
10    100.541436
11     95.071823
12     99.138122
13    112.674033
14    100.425414
15     92.646409
16     96.801105
17    104.033149
18    113.232044
19    123.519337
20    143.662983
21    151.055249
22    160.022099
23    164.806630
Name: occupancy, dtype: float64

In [48]:
import pandas as pd
inf = pd.read_csv("../dataset/UrbanEV/inf.csv")

inf.head()


,station_id,longitude,latitude,charge_count,TAZID,area,perimeter
0,1001,113.784724,22.714121,20,559,5726523.056,9624.5755
1,1002,113.785002,22.725900,22,558,3927542.045,8189.3687
2,1003,113.787971,22.735538,6,558,3927542.045,8189.3687
3,1004,113.788126,22.693449,11,596,2972113.241,7335.1002
4,1006,113.790987,22.731351,10,594,2044150.527,5825.1986


In [49]:
zone_charge = (
    inf
    .groupby("TAZID")["charge_count"]
    .sum()
)

zone_charge.loc[1167]

np.int64(373)

In [50]:
#对于不知道1167区域的充电桩是否是多的
zone_charge.describe()

count    275.000000
mean      63.752727
std       56.733632
min        4.000000
25%       24.000000
50%       48.000000
75%       84.000000
max      373.000000
Name: charge_count, dtype: float64

In [51]:
zone_charge.sort_values(ascending=False).head(10)

TAZID
1167    373
1011    332
526     262
329     254
335     249
703     233
1068    227
519     223
1009    207
1049    200
Name: charge_count, dtype: int64

In [52]:
poi = pd.read_csv("../dataset/UrbanEV/poi.csv")

poi.info()

poi.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 712135 entries, 0 to 712134
Data columns (total 3 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   primary_types  712135 non-null  object 
 1   longitude      712135 non-null  float64
 2   latitude       712135 non-null  float64
dtypes: float64(2), object(1)
memory usage: 16.3+ MB


,primary_types,longitude,latitude
0,lifestyle services,114.576628,22.494887
1,lifestyle services,114.579614,22.492192
2,lifestyle services,114.565933,22.500528
3,lifestyle services,114.577400,22.492460
4,lifestyle services,114.574730,22.494576


In [53]:
import pandas as pd
poi = pd.read_csv("../dataset/UrbanEV/poi.csv")
poi["primary_types"].value_counts()

primary_types
lifestyle services            393428
business and residential      183383
food and beverage services    135324
Name: count, dtype: int64

In [54]:
import os
os.environ['SHAPE_RESTORE_SHX'] = 'YES'

districts = gpd.read_file("../dataset/SZ_districts/SZ_districts.shp")
print(districts.crs)
print(districts.head())

EPSG:3857
   OBJECTID   TAZID   AREA           X          Y   ZONE  LENG_ROAD  \
0         1  1054.0  11.49  114.333530  22.783052  781.0     10.135   
1         2  1052.0  17.07  114.282465  22.789111  775.0      6.368   
2         3   850.0   2.85  114.322722  22.791482  780.0      4.822   
3         4  1066.0   1.57  113.910301  22.521287  307.0      7.130   
4         5  1068.0   1.41  113.918680  22.517837  306.0      7.084   

     SHAPE_Leng  SHAPE_Area                                           geometry  
0  25856.846986  13581804.0  POLYGON ((12727715 2609219, 12727838 2609098, ...  
1  30117.841314  20191527.5  POLYGON ((12721609 2609078, 12721630 2608932, ...  
2  10307.116136   3375499.0  POLYGON ((12726477 2609132, 12726623 2609056, ...  
3   5552.707185   1857995.0  POLYGON ((12680558 2573496, 12680096 2573463, ...  
4   5930.667262   1654644.0  POLYGON ((12681747 2573552, 12681409 2572621, ...  


In [57]:
districts["TAZID"] = districts["TAZID"].astype(int)

inf["TAZID"] = inf["TAZID"].astype(int)

occupancy_long["zone"] = occupancy_long["zone"].astype(int)

In [58]:
print(districts["TAZID"].dtype)
print(inf["TAZID"].dtype)
print(occupancy_long["zone"].dtype)

int64
int64
int64


In [59]:
zone_occupancy = (
    occupancy_long
    .groupby("zone")["occupancy"]
    .mean()
    .reset_index()
)


zone_occupancy.head()

,zone,occupancy
0,102,14.207873
1,104,1.196478
2,105,17.774056
3,106,49.159876
4,107,17.214434


In [61]:
zone_occupancy.rename(
    columns={"zone":"TAZID"},
    inplace=True
)
zone_occupancy

,TAZID,occupancy
0,102,14.207873
1,104,1.196478
2,105,17.774056
3,106,49.159876
4,107,17.214434
...,...,...
270,1166,23.478591
271,1167,137.304558
272,1168,8.238605
273,1172,3.131906


In [62]:
zone_occupancy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 275 entries, 0 to 274
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   TAZID      275 non-null    int64  
 1   occupancy  275 non-null    float64
dtypes: float64(1), int64(1)
memory usage: 4.4 KB


In [63]:
zone_occupancy.to_csv(
    "../dataset/processed/zone_occupancy.csv",
    index=False
)

print("保存完成")

保存完成


In [ ]:
# 项目目标

## 本阶段解决的问题

官方UrbanEV数据集提供了区域级occupancy ratio指标（单位%），
但未公开该指标的具体计算公式。

本阶段主要完成occupancy数据的数据整理与区域级统计分析，
将原始时间序列数据转换为适合空间分析的数据格式。


## 使用数据

数据来源：

UrbanEV公开数据集（GitHub）

使用文件：

- occupancy.csv
- SZ_districts.shp


## 当前阶段输出

1. 清洗后的occupancy长表

2. 区域级occupancy统计结果表

3. 为后续空间可视化分析提供的数据基础